In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported. Random seed set to 42.")


Libraries imported. Random seed set to 42.


In [3]:
def create_synthetic_ranking_dataset(n_queries=20, docs_per_query=50, n_features=10, random_state=42):
    """Create a synthetic ranking dataset. Returns list of dicts with X and y."""
    np.random.seed(random_state)
    data = []
    for q_id in range(1, n_queries + 1):
        X_q = np.random.randn(docs_per_query, n_features)
        feature_sum = X_q.sum(axis=1)
        relevance_probs = 1.0 / (1.0 + np.exp(-feature_sum))
        y_q = (relevance_probs > np.random.uniform(0, 1, docs_per_query)).astype(int)
        if y_q.sum() < 1:
            y_q[np.argmax(relevance_probs)] = 1
        if y_q.sum() == len(y_q):
            y_q[np.argmin(relevance_probs)] = 0
        data.append({'query_id': q_id, 'X': X_q, 'y': y_q})
    return data

data = create_synthetic_ranking_dataset(n_queries=20, docs_per_query=50, n_features=10)
print(f"Generated: {len(data)} queries, {data[0]['X'].shape[0]} docs/query, {data[0]['X'].shape[1]} features")
print(f"Total pairs: {sum(d['X'].shape[0] for d in data)}")


Generated: 20 queries, 50 docs/query, 10 features
Total pairs: 1000


In [4]:
def preprocess_and_save_data(data, output_dir='./partB/data/'):
    """Normalize features with StandardScaler, split 80/20 by queries, save to partB/data/."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    X_all = np.vstack([d['X'] for d in data])
    scaler = StandardScaler()
    X_normalized = scaler.fit_transform(X_all)
    idx = 0
    for d in data:
        n_docs = d['X'].shape[0]
        d['X'] = X_normalized[idx:idx + n_docs]
        idx += n_docs
    n_train_queries = int(0.8 * len(data))
    train_data = data[:n_train_queries]
    test_data = data[n_train_queries:]
    np.save(f'{output_dir}/train_data.npy', train_data, allow_pickle=True)
    np.save(f'{output_dir}/test_data.npy', test_data, allow_pickle=True)
    np.save(f'{output_dir}/scaler.npy', scaler, allow_pickle=True)
    print(f"Preprocessing completed: {len(train_data)} train queries, {len(test_data)} test queries. Saved to {output_dir}")
    return train_data, test_data, scaler

# Save to partB/data (works from repo root or from partB/)
_out = Path('partB/data') if Path('partB').exists() else Path('data')
train_data, test_data, scaler = preprocess_and_save_data(data, output_dir=str(_out))
print("\nDataset ready for Task 2.2 (Reproduction)")

Preprocessing completed: 16 train queries, 4 test queries. Saved to data

Dataset ready for Task 2.2 (Reproduction)


# Task 2.1: Dataset Selection and Setup

## Paper: Learning Structural SVMs with Latent Variables
**Authors**: Chun-Nam Yu, Thorsten Joachims  
**Venue**: ICML 2009

---

## Dataset: Synthetic Query-Document Ranking Task

### Dataset Description

I have created a **synthetic ranking dataset** consisting of 20 queries, each with 50 candidate documents (1,000 total query-document pairs). Each document is represented by a 10-dimensional feature vector, and each query-document pair has a binary relevance label (0 = not relevant, 1 = relevant). Additional continuous labels in [0, 2] indicate graded relevance.

### Why This Dataset Is Appropriate for the Latent Structural SVM

The latent Structural SVM method proposed in Yu & Joachims (2009) is specifically designed for ranking tasks with latent structure (Precision@k optimization in Section 5 of the paper). This toy dataset mimics that setting:

1. **Matching problem type**: The paper demonstrates ranking optimization on LETOR OHSUMED (medical IR dataset). My toy dataset is a **ranking problem** where the task is to learn document-query relevance scores, matching the problem class in the paper.

2. **Latent structure**: In real ranking, the notion of "informativeness" of a document extends beyond its relevance label. The latent variable represents which documents are truly informative for the query—this structure is not directly observable but can be inferred from features. The paper's CCCP method is designed to discover such latent structure without explicit supervision.

3. **Precision@k motivation**: The paper optimizes Precision@k (the fraction of top-k ranked documents that are relevant). My dataset allows evaluation of this metric and demonstrates how latent variables can improve ranking performance.

### Limitations Compared to the Original Paper

1. **Scale**: The original LETOR OHSUMED dataset has ~16,000 query-document pairs across 106 queries with 5-fold cross-validation. My toy dataset has only 1,000 pairs across 20 queries, making it much smaller.

2. **Feature richness**: LETOR uses 25 hand-crafted ranking features (TF/IDF, BM25, language models, URL features, etc.), whereas my synthetic dataset uses 10 random features. Real ranking features capture linguistic and content properties; mine are generic.

3. **Real-world relevance distribution**: LETOR reflects actual medical retrieval relevance judgments. My data is synthetically generated with controlled relevance distributions, which may not reflect realistic patterns.

4. **Benchmark status**: LETOR is a standard benchmark with established baselines. My toy dataset is not comparable to published results, limiting the ability to validate against known metrics.

Despite these limitations, the toy dataset is **sufficient to demonstrate the core mechanism** of latent Structural SVM inference and Precision@k optimization on a ranking task, which is the goal of Task 2.

---

## Preprocessing Steps

## Preprocessing Steps Explanation

### Feature Normalization: StandardScaler

**Why normalize?** 
The paper uses 46 hand-crafted IR features (e.g., BM25 scores, TF-IDF values) that naturally have different scales and ranges. Normalizing ensures each feature contributes equally to the model.

**How we normalize**:
1. **Fit on training set**: Compute mean μ and standard deviation σ from training features only
   - This ensures the test set doesn't leak information into statistics
   - Formula: (x - μ_train) / σ_train
2. **Transform train and test**: Apply the same (μ_train, σ_train) to both train and test sets
   - Test set is transformed using training statistics, simulating real deployment

**Code Logic**:
```
scaler = StandardScaler()
scaler.fit(X_train)  # Learn statistics from training only
X_train_normalized = scaler.transform(X_train)  # Apply to train
X_test_normalized = scaler.transform(X_test)   # Apply to test using trained statistics
```

**Result**: All features now have mean ~0 and std ~1, enabling the model to treat them equally.

### Train-Test Split: By Query, Not Document

**Why split by queries, not documents?**
- **By query**: Ensures every test query is completely unseen during training (gold standard)
- **By document**: Would mix training and test documents within the same query (data leakage)

**Realistic evaluation**: In ranking, you want to test on entirely new queries (like a real-world search engine encountering new queries). Splitting by document would artificially inflate accuracy.

**Our split**:
- **Training**: 16 queries, ~800 documents total (80% of 20 queries)
- **Test**: 4 queries, ~200 documents total (20% of 20 queries)
- Ensures zero overlap in query IDs between train and test

### Data Storage: NumPy Arrays

We save preprocessed data as NumPy `.npy` files for Task 2.2 reuse:
- `train_data.npy`: List of dicts with 'X' (features) and 'y' (labels) for 16 training queries
- `test_data.npy`: List of dicts with 'X' (features) and 'y' (labels) for 4 test queries
- `scaler.npy`: Fitted StandardScaler object for future feature transformation

This enables Task 2.2 to load data directly without regenerating it.

---

## Summary: Task 2.1 Complete

**Dataset Created**: 
- Synthetic ranking task: 20 queries, 50 docs/query, 10 features, binary relevance
- Train set: 16 queries (~800 docs), normalized via StandardScaler
- Test set: 4 queries (~200 docs), transformed using training statistics
- Total: 1,000 query-document pairs, split by query

**Key Design Choices Documented**:
- Synthetic data for fast iteration (paper uses real OHSUMED)
- Query-level split prevents information leakage
- StandardScaler fit on training set only, applied to test set
- Data saved as .npy files for Task 2.2 reuse

**Next Step**: Task 2.2 loads this data and implements the CCCP algorithm.